# 09 — Combined accepted pipeline plus stemming

Iteration 8 keeps document identification, focused clause matching, and zero-match fallback. It adds only English Snowball stemming to document/query tokens. Examples: license/licenses/licensing → licens, assignment/assign → assign. Stemming does not recognize all synonyms or establish legal equivalence.

Stored contract text and offsets remain unchanged. No LLM, embeddings, FAISS, clause-aware chunking, or test evaluation is used. Running this notebook reviews the saved run without advancing the loop.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from improve import read_log, read_json
from retrieval import BM25Retriever, load_inputs

run = next(r for r in reversed(read_log(ROOT / 'experiments/loop_log.jsonl')) if r['iteration']==8)
print('Change:', run['changes'])
print('Accepted:', run['decision']['accepted'])
rows=[]
for label, result in run['metrics'].items():
    at5=next(m for m in result['retrieval'] if m['k']==5)
    rows.append(dict(approach=label, precision=at5['precision'], recall=at5['recall'],
                     benchmark_precision=at5['benchmark_precision'], benchmark_recall=at5['benchmark_recall'],
                     false_abstention=result['false_abstention'], correct_abstention=result['correct_abstention'],
                     p95_ms=result['latency_ms']['p95']))
display(pd.DataFrame(rows))
display(pd.Series(run['decision']['checks'],name='Passed'))
print('Before:',run['diagnosis_before'])
print('After:',run['diagnosis_after'])

Change: {'stemming': {'before': None, 'after': 'english'}}
Accepted: True


,approach,precision,recall,benchmark_precision,benchmark_recall,false_abstention,correct_abstention,p95_ms
0,baseline,0.017433,0.181647,0.017594,0.192427,0.0,0.0,8.47192
1,reference,0.062782,0.535021,0.065070,0.591568,0.0,0.0,10.98969
2,candidate,0.078798,0.670075,0.083766,0.765975,0.0,0.0,10.49893


recall_gain                True
precision                  True
answerable_abstention      True
unanswerable_abstention    True
latency                    True
Name: Passed, dtype: bool

Before: {'complete_coverage': 57, 'right_document_wrong_section': 50, 'partial_coverage': 33, 'false_returns': 23}
After: {'complete_coverage': 82, 'right_document_wrong_section': 36, 'partial_coverage': 22}


## Run baseline, previous accepted model, or new model

Change `which` and edit the question. Routing uses the `Consider ...; question` form; other phrasing uses the original global full-query behavior. Stemming requires NLTK but no language-data download.

In [2]:
which='candidate_config'
configs={'baseline':read_json(ROOT/'configs/bm25_baseline.json'),
         'reference_config':run['reference_config'], 'candidate_config':run['candidate_config']}
_,documents,_=load_inputs(ROOT)
model=BM25Retriever(documents,configs[which])
question='Consider the Wireless Content License Agreement between Fox Mobile Entertainment and Glu Mobile; What is the expiration date of this contract?'
hits=model.search(question,k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,4.913346,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,1.920903,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


## Explain this step

“We kept the two working query changes and normalized word endings in both the index and the query. That helped BM25 match related word forms without changing original evidence text. The combined candidate passed the same predeclared acceptance checks.”

Unique-character recall@5 is **18.16% baseline → 53.50% previous → 67.01% current**. Precision@5 is **1.74% → 6.28% → 7.88%**. Complete gold coverage rose from 57 to 82 of 140 train questions. Reference-compatible current recall is 76.60%, which counts repeated overlap differently.

This is the third accepted improvement in eight distinct non-LLM trials. It remains train evidence, not proof of generalization. Current correct negative abstention is still 0/23. The system still needs reliable answerability, the required interfaces, and final held-out evaluation. Two slots remain under the user's ten-iteration limit.